In [55]:
import pandas as pd
import numpy as np
import nolds
import matplotlib.pyplot as plt

In [26]:
def parse_tensor_string(tensor_str):
    # Remove 'tensor', '(', ')', and convert to list
    return np.squeeze(np.array(eval(tensor_str.replace("tensor", "").replace("(", "").replace(")", ""))), axis=-1)

In [31]:
tau = 60
x0 = 0.2
frac = 30
location = "last"
datasetname = f"MG_x0_{x0}_tau_{tau}_{frac}_{location}"
df = pd.read_csv(f"results/{datasetname}/dset_test.csv")
df['past_target'] = df['past_target'].apply(parse_tensor_string)
df['future_target'] = df['future_target'].apply(parse_tensor_string)
past_targets = np.stack(df['past_target'].values)
future_targets = np.stack(df['future_target'].values)
pred = np.squeeze(np.load(f"results/{datasetname}/predictions_original.npy"),axis = -1)
pred_per = np.squeeze(np.load(f"results/{datasetname}/predictions_perturbed.npy"),axis = -1)


In [37]:
mse_original = np.mean((pred - future_targets) ** 2)
mse_perturbed = np.mean((pred_per - future_targets) ** 2)
print(f"Original MSE: {mse_original}")
print(f"Perturbed MSE: {mse_perturbed}")

Original MSE: 0.408062583538246
Perturbed MSE: 0.408062583538246


In [52]:
idx = 40

In [ ]:
pred_lyap = nolds.lyap_r(pred[idx], emb_dim=10, min_tsep=33)
pred_per_lyap = nolds.lyap_r(pred_per[idx], emb_dim=10, min_tsep=33)
true_lyap = nolds.lyap_r(future_targets[idx], emb_dim=10, min_tsep=33)
past_lyap = nolds.lyap_r(past_targets[idx][-96:], emb_dim=10, min_tsep=33)
print(f"Original Lyapunov Exponent: {pred_lyap}")
print(f"Perturbed Lyapunov Exponent: {pred_per_lyap}")
print(f"True Lyapunov Exponent: {true_lyap}")
print(f"Past Lyapunov Exponent: {past_lyap}")

In [ ]:
t_full = np.arange(608)
t_pred = np.arange(512, 608) 

plt.figure(figsize=(12, 5))
plt.plot(t_full[:-96], past_targets[idx], label='Observed', color='blue')     
plt.plot(t_pred, future_targets[idx], label='True Value', color='green') 
plt.plot(t_pred, pred[idx], label='Prediction', color='red', linestyle='--')      
plt.plot(t_pred, pred_per[idx], label='Prediction_Perturbed', color='black', linestyle='--') 

plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Time Series Forecasting')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def delay_embedding_3d(series, delay=1):
    return np.column_stack([
        series[:-2 * delay],
        series[delay:-delay],
        series[2 * delay:]
    ])

# Parameters
delay = 1  # You can try 2, 3, etc.

# Ensure both series are aligned in length for delay embedding
# Use the last 96 + 2*delay from ground truth to match prediction size
gt_segment = np.concatenate([past_targets[idx],future_targets[idx]])
pred_segment = pred[idx]

# Generate embeddings
gt_embed = delay_embedding_3d(gt_segment, delay)
pred_embed = delay_embedding_3d(pred_segment, delay)
pred_per_embed = delay_embedding_3d(pred_per[idx], delay)

# Plotting
fig = plt.figure(figsize=(12, 7))
ax = fig.add_subplot(111, projection='3d')

# Plot ground truth
ax.plot(gt_embed[:, 0], gt_embed[:, 1], gt_embed[:, 2], label='Ground Truth', alpha=0.4)

# Plot prediction
ax.plot(pred_embed[:, 0], pred_embed[:, 1], pred_embed[:, 2], label='Prediction', alpha=0.7)

# Plot perturbed prediction
ax.plot(pred_per_embed[:, 0], pred_per_embed[:, 1], pred_per_embed[:, 2], label='Perturbed Prediction', alpha=0.2)

# Labels and styling
ax.set_xlabel(f'x(t)')
ax.set_ylabel(f'x(t+{delay})')
ax.set_zlabel(f'x(t+{2*delay})')
ax.set_title('3D Delay Embedding')
ax.legend()
plt.tight_layout()
plt.show()

